In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [32]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

events = pd.read_csv(DATA_DIR / "events.csv")
category_tree = pd.read_csv(DATA_DIR / "category_tree.csv")

In [33]:
events.head(10)

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN
5,1433224086234,972639,view,22556,NaN
6,1433221923240,810725,view,443030,NaN
7,1433223291897,794181,view,439202,NaN
8,1433220899221,824915,view,428805,NaN
9,1433221204592,339335,view,82389,NaN


In [34]:
category_tree.head(10)

,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0
3,1691,885.0
4,536,1691.0
5,231,NaN
6,542,378.0
7,1146,542.0
8,1140,542.0
9,1479,1537.0


In [35]:
print(f"Events dataset shape: {events.shape}")
print(f"Category tree dataset shape: {category_tree.shape}")

Events dataset shape: (2756101, 5)
Category tree dataset shape: (1669, 2)


In [36]:
events.describe(include='all')

,timestamp,visitorid,event,itemid,transactionid
count,2.756101e+06,2.756101e+06,2756101,2.756101e+06,22457.000000
unique,NaN,NaN,3,NaN,NaN
top,NaN,NaN,view,NaN,NaN
freq,NaN,NaN,2664312,NaN,NaN
mean,1.436424e+12,7.019229e+05,NaN,2.349225e+05,8826.497796
std,3.366312e+09,4.056875e+05,NaN,1.341954e+05,5098.996290
min,1.430622e+12,0.000000e+00,NaN,3.000000e+00,0.000000
25%,1.433478e+12,3.505660e+05,NaN,1.181200e+05,4411.000000
50%,1.436453e+12,7.020600e+05,NaN,2.360670e+05,8813.000000
75%,1.439225e+12,1.053437e+06,NaN,3.507150e+05,13224.000000


In [37]:
category_tree.describe(include='all')

,categoryid,parentid
count,1669.000000,1644.000000
mean,849.285201,847.571168
std,490.195116,505.058485
min,0.000000,8.000000
25%,427.000000,381.000000
50%,848.000000,866.000000
75%,1273.000000,1291.000000
max,1698.000000,1698.000000


In [38]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          object 
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), object(1)
memory usage: 105.1+ MB


In [39]:
category_tree.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1669 entries, 0 to 1668
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   categoryid  1669 non-null   int64  
 1   parentid    1644 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 26.2 KB


In [40]:
total_number_of_events = len(events)
print(f"Total number of events: {total_number_of_events}")

Total number of events: 2756101


In [41]:
unique_visitors = events['visitorid'].nunique()
print(f"Unique visitors: {unique_visitors}")

Unique visitors: 1407580


In [42]:
number_of_unique_products = events['itemid'].nunique()
print(f"Number of unique products: {number_of_unique_products}")

Number of unique products: 235061


In [43]:
number_of_unique_transactions = events['transactionid'].nunique()
print(f"Number of unique transactions: {number_of_unique_transactions}")

Number of unique transactions: 17672


In [44]:
number_of_each_event_type = events['event'].value_counts().to_frame(name='event_count')

number_of_each_event_type['percentage'] = (
    events['event'].value_counts(normalize=True) * 100
).round(2)

print("Number of each event type:")
display(number_of_each_event_type)

Number of each event type:


,event_count,percentage
event,,
view,2664312,96.67
addtocart,69332,2.52
transaction,22457,0.81


In [45]:
events.isna().sum()

timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

In [46]:
events.isna().sum() / len(events) * 100

timestamp         0.00000
visitorid         0.00000
event             0.00000
itemid            0.00000
transactionid    99.18519
dtype: float64

In [47]:
data_quality_report = pd.DataFrame({
    'null_count': events.isnull().sum(),
    'null_percentage': (events.isnull().mean() * 100).round(2),
    'unique_values': events.nunique(),
    'duplicated_count': [
        events.duplicated(subset=[col]).sum()
        for col in events.columns
    ]
})

display(data_quality_report)

,null_count,null_percentage,unique_values,duplicated_count
timestamp,0,0.00,2750455,5646
visitorid,0,0.00,1407580,1348521
event,0,0.00,3,2756098
itemid,0,0.00,235061,2521040
transactionid,2733644,99.19,17672,2738428


In [48]:
event_metrics = {
    "total_events": len(events),
    "unique_visitors": events["visitorid"].nunique(),
    "unique_items": events["itemid"].nunique(),
    "unique_transactions": events["transactionid"].nunique()
}

display(pd.Series(event_metrics, name="value").to_frame())

,value
total_events,2756101
unique_visitors,1407580
unique_items,235061
unique_transactions,17672


In [49]:
def get_data_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    report = pd.DataFrame({
        "null_count": df.isna().sum(),
        "null_percentage": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(),
    })

    return report


events_quality = get_data_quality_report(events)
categories_quality = get_data_quality_report(category_tree)

print("EVENTS DATA QUALITY")
display(events_quality)

print("CATEGORY TREE DATA QUALITY")
display(categories_quality)

EVENTS DATA QUALITY


,null_count,null_percentage,unique_values
timestamp,0,0.00,2750455
visitorid,0,0.00,1407580
event,0,0.00,3
itemid,0,0.00,235061
transactionid,2733644,99.19,17672


CATEGORY TREE DATA QUALITY


,null_count,null_percentage,unique_values
categoryid,0,0.0,1669
parentid,25,1.5,362


In [50]:
events_duplicates = events.duplicated().sum()

events_duplicates_percentage = (
    events.duplicated().mean() * 100
).round(2)

categories_duplicates = category_tree.duplicated().sum()

categories_duplicates_percentage = (
    category_tree.duplicated().mean() * 100
).round(2)

duplicates_report = pd.DataFrame({
    "dataset": ["events", "category_tree"],
    "duplicated_rows": [
        events_duplicates,
        categories_duplicates
    ],
    "duplicated_percentage": [
        events_duplicates_percentage,
        categories_duplicates_percentage
    ]
})

display(duplicates_report)

,dataset,duplicated_rows,duplicated_percentage
0,events,460,0.02
1,category_tree,0,0.00


In [51]:
events["event_datetime"] = pd.to_datetime(
    events["timestamp"],
    unit="ms",
    utc=True,
    errors="coerce"
)

events["event_date"] = events["event_datetime"].dt.date
events["event_hour"] = events["event_datetime"].dt.hour
events["event_weekday"] = events["event_datetime"].dt.day_name()

print(f"Minimum date: {events['event_datetime'].min()}")
print(f"Maximum date: {events['event_datetime'].max()}")

print(
    f"Unique event dates: {events['event_date'].nunique()}"
)

Minimum date: 2015-05-03 03:00:04.384000+00:00
Maximum date: 2015-09-18 02:59:47.788000+00:00
Unique event dates: 139


In [52]:
invalid_timestamps = events["event_datetime"].isna().sum()

negative_timestamps = (events["timestamp"] < 0).sum()

timestamp_quality = pd.DataFrame({
    "check": [
        "Invalid timestamps",
        "Negative timestamps"
    ],
    "count": [
        invalid_timestamps,
        negative_timestamps
    ]
})

display(timestamp_quality)

,check,count
0,Invalid timestamps,0
1,Negative timestamps,0


In [53]:
daily_events = (
    events.groupby("event_date")
    .size()
    .reset_index(name="event_count")
)

display(daily_events.head())

plt.figure(figsize=(15, 5))

sns.lineplot(
    data=daily_events,
    x="event_date",
    y="event_count"
)

plt.title("Daily Event Volume")
plt.xlabel("Date")
plt.ylabel("Number of Events")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

,event_date,event_count
0,2015-05-03,13683
1,2015-05-04,19414
2,2015-05-05,23015
3,2015-05-06,23920
4,2015-05-07,23164


NameError: name 'sns' is not defined

<Figure size 1500x500 with 0 Axes>

In [54]:
hourly_events = (
    events.groupby("event_hour")
    .size()
    .reset_index(name="event_count")
)

plt.figure(figsize=(12, 5))

sns.barplot(
    data=hourly_events,
    x="event_hour",
    y="event_count"
)

plt.title("Event Volume by Hour (UTC)")
plt.xlabel("Hour")
plt.ylabel("Number of Events")
plt.tight_layout()
plt.show()

NameError: name 'sns' is not defined

<Figure size 1200x500 with 0 Axes>

In [55]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_events = (
    events["event_weekday"]
    .value_counts()
    .reindex(weekday_order)
    .fillna(0)
)

display(weekday_events.to_frame(name="event_count"))

,event_count
event_weekday,
Monday,439813
Tuesday,447077
Wednesday,431114
Thursday,418761
Friday,379699
Saturday,305215
Sunday,334422


In [56]:
category_metrics = {
    "total_categories": len(category_tree),
    "unique_category_ids": category_tree["categoryid"].nunique(),
    "root_categories": category_tree["parentid"].isna().sum(),
    "categories_with_parent": category_tree["parentid"].notna().sum(),
    "duplicate_category_ids": category_tree["categoryid"].duplicated().sum()
}

display(
    pd.Series(category_metrics, name="value").to_frame()
)

,value
total_categories,1669
unique_category_ids,1669
root_categories,25
categories_with_parent,1644
duplicate_category_ids,0


In [57]:
category_ids = set(category_tree["categoryid"].dropna())

categories_with_parent = category_tree[
    category_tree["parentid"].notna()
]

missing_parent_categories = categories_with_parent[
    ~categories_with_parent["parentid"].isin(category_ids)
]

print(
    f"Categories with missing parent: "
    f"{len(missing_parent_categories)}"
)

display(missing_parent_categories.head(20))

Categories with missing parent: 0


,categoryid,parentid


In [58]:
duplicate_category_ids = category_tree[
    category_tree["categoryid"].duplicated(keep=False)
].sort_values("categoryid")

display(duplicate_category_ids)

,categoryid,parentid


In [59]:
expected_event_types = {
    "view",
    "addtocart",
    "transaction"
}

unknown_event_types = events[
    ~events["event"].isin(expected_event_types)
]

print(f"Unknown event types: {len(unknown_event_types)}")
display(unknown_event_types.head())

Unknown event types: 0


,timestamp,visitorid,event,itemid,transactionid,event_datetime,event_date,event_hour,event_weekday


In [60]:
transaction_null_analysis = (
    events.groupby("event")
    .agg(
        total_events=("event", "size"),
        null_transaction_ids=(
            "transactionid",
            lambda x: x.isna().sum()
        )
    )
)

transaction_null_analysis["null_percentage"] = (
    transaction_null_analysis["null_transaction_ids"]
    / transaction_null_analysis["total_events"]
    * 100
).round(2)

display(transaction_null_analysis)

,total_events,null_transaction_ids,null_percentage
event,,,
addtocart,69332,69332,100.0
transaction,22457,0,0.0
view,2664312,2664312,100.0


In [61]:
purchases_without_id = events[
    (events["event"] == "transaction")
    & (events["transactionid"].isna())
]

print(
    f"Purchases without transaction ID: "
    f"{len(purchases_without_id)}"
)

Purchases without transaction ID: 0


In [62]:
purchases_without_id = events[
    (events["event"] == "transaction")
    & (events["transactionid"].isna())
]

print(
    f"Purchases without transaction ID: "
    f"{len(purchases_without_id)}"
)

Purchases without transaction ID: 0


In [63]:
visitor_item_repetitions = (
    events.groupby(["visitorid", "itemid"])
    .size()
    .reset_index(name="event_count")
)

repeated_visitor_item_pairs = visitor_item_repetitions[
    visitor_item_repetitions["event_count"] > 1
]

print(
    f"Visitor-item pairs with multiple events: "
    f"{len(repeated_visitor_item_pairs)}"
)

display(
    repeated_visitor_item_pairs
    .sort_values("event_count", ascending=False)
    .head(10)
)

Visitor-item pairs with multiple events: 333038


,visitorid,itemid,event_count
595525,388556,306289,308
1941998,1272794,413901,233
2088249,1369328,356339,214
1246521,816229,396064,180
435521,285368,40870,177
1793398,1172198,414460,176
2056658,1348163,191049,170
48212,32252,133138,170
82051,54791,296619,161
1677386,1098890,6605,157


In [64]:
transaction_event_counts = (
    events[events["event"] == "transaction"]
    .groupby("transactionid")
    .size()
    .reset_index(name="items_per_transaction")
)

display(transaction_event_counts.describe())

print(
    "Transactions with multiple event rows:",
    (transaction_event_counts["items_per_transaction"] > 1).sum()
)

,transactionid,items_per_transaction
count,17672.000000,17672.000000
mean,8835.500000,1.270767
std,5101.611314,0.974793
min,0.000000,1.000000
25%,4417.750000,1.000000
50%,8835.500000,1.000000
75%,13253.250000,1.000000
max,17671.000000,31.000000


Transactions with multiple event rows: 2710


In [65]:
dq_metrics = pd.DataFrame([
    {
        "metric_id": "DQ-01",
        "metric": "Total events",
        "value": len(events)
    },
    {
        "metric_id": "DQ-02",
        "metric": "Unique visitors",
        "value": events["visitorid"].nunique()
    },
    {
        "metric_id": "DQ-03",
        "metric": "Unique items",
        "value": events["itemid"].nunique()
    },
    {
        "metric_id": "DQ-04",
        "metric": "Unique transactions",
        "value": events["transactionid"].nunique()
    },
    {
        "metric_id": "DQ-05",
        "metric": "Total NULL values in events",
        "value": events.isna().sum().sum()
    },
    {
        "metric_id": "DQ-06",
        "metric": "Fully duplicated event rows",
        "value": events.duplicated().sum()
    },
    {
        "metric_id": "DQ-07",
        "metric": "Distinct event types",
        "value": events["event"].nunique()
    },
    {
        "metric_id": "DQ-08",
        "metric": "Observation period",
        "value": (
            f"{events['event_datetime'].min()} — "
            f"{events['event_datetime'].max()}"
        )
    },
    {
        "metric_id": "DQ-09",
        "metric": "Distinct event dates",
        "value": events["event_date"].nunique()
    },
    {
        "metric_id": "DQ-10",
        "metric": "Total categories",
        "value": category_tree["categoryid"].nunique()
    }
])

display(dq_metrics)

,metric_id,metric,value
0,DQ-01,Total events,2756101
1,DQ-02,Unique visitors,1407580
2,DQ-03,Unique items,235061
3,DQ-04,Unique transactions,17672
4,DQ-05,Total NULL values in events,2733644
5,DQ-06,Fully duplicated event rows,460
6,DQ-07,Distinct event types,3
7,DQ-08,Observation period,2015-05-03 03:00:04.384000+00:00 — 2015-09-18 ...
8,DQ-09,Distinct event dates,139
9,DQ-10,Total categories,1669
